In [3]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 字体设置：新罗马字体，黑色，10.5号
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10.5
plt.rcParams['text.color'] = 'black'

# 创建基础网格 (原始状态的正方形板子)
x = np.linspace(-1, 1, 15)
y = np.linspace(-1, 1, 15)
X, Y = np.meshgrid(x, y)
Z0 = np.zeros_like(X)

# 2. 配色设置：6种不同的莫兰迪浅色色系
morandi_colors = [
    '#D8C6B8',  # N11: 浅灰驼色
    '#B5C4B1',  # N22: 浅豆沙绿
    '#AAB5C1',  # N66: 浅雾霾蓝
    '#D1BDBE',  # M11: 浅灰粉色
    '#E0D4C3',  # M22: 浅燕麦色
    '#C4C5CD'   # M66: 浅丁香灰
]
edge_color = '#8A8A8A'  # 统一的网格线颜色，偏柔和的灰色

def save_mode_plot(filename, Xd, Yd, Zd, surface_color):
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection='3d')

    # 绘制原始形状 (灰色虚线网格)
    ax.plot_wireframe(X, Y, Z0, color='gray', linestyle='--', alpha=0.35)

    # 绘制变形后的形状 (莫兰迪色系实心曲面)
    ax.plot_surface(Xd, Yd, Zd, alpha=0.9, color=surface_color, edgecolor=edge_color, linewidth=0.5)

    # 固定坐标轴范围，保证视角和比例统一
    ax.set_xlim([-1.25, 1.25])
    ax.set_ylim([-1.25, 1.25])
    ax.set_zlim([-0.6, 0.6])

    # 隐藏坐标轴让图像更清爽
    ax.axis('off')

    plt.tight_layout()
    # 导出背景透明的无损高分辨率图片
    plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.close()

# ----------------- 生成并保存 6 张图片 -----------------
# 3. 变形减小：降低了所有方向的拉伸与曲率系数
# 4. 去除Title和箭头：直接传入对应的变形矩阵即可

# N11: X方向的面内拉伸
save_mode_plot('N11.png', X*1.12, Y*0.94, Z0, morandi_colors[0])

# N22: Y方向的面内拉伸
save_mode_plot('N22.png', X*0.94, Y*1.12, Z0, morandi_colors[1])

# N66: 面内剪切
save_mode_plot('N66.png', X + 0.15*Y, Y + 0.15*X, Z0, morandi_colors[2])

# M11: 绕Y轴的弯矩 (X向曲率)
save_mode_plot('M11.png', X, Y, 0.2*X**2, morandi_colors[3])

# M22: 绕X轴的弯矩 (Y向曲率)
save_mode_plot('M22.png', X, Y, 0.2*Y**2, morandi_colors[4])

# M66: 面内扭矩
save_mode_plot('M66.png', X, Y, 0.25*X*Y, morandi_colors[5])

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 全局基础设置
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10.5
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.unicode_minus'] = False

# 板的物理半厚度（用于构建高保真实体模型）
h_2 = 0.08

# 莫兰迪色系字典
morandi_colors = {
    'B': '#B5C4B1',  # 浅豆沙绿 ([A]矩阵：面内刚度)
    'A': '#D1BDBE',  # 浅灰粉色 ([B]矩阵：耦合刚度)
    'D': '#AAB5C1'   # 浅雾霾蓝 ([D]矩阵：抗弯刚度)
}

# -------------------- 几何构建模块 --------------------
# 获取原始正方体的 12 条边界线框 (用于参照)
def get_edges():
    t = np.linspace(-1, 1, 40)
    ones = np.ones_like(t)
    return [
        (t, ones, h_2*ones), (t, -ones, h_2*ones), (ones, t, h_2*ones), (-ones, t, h_2*ones),
        (t, ones, -h_2*ones), (t, -ones, -h_2*ones), (ones, t, -h_2*ones), (-ones, t, -h_2*ones),
        (ones, ones, t*h_2), (ones, -ones, t*h_2), (-ones, ones, t*h_2), (-ones, -ones, t*h_2)
    ]

# 获取原始长方体的 6 个表面面域 (用于渲染实体)
def get_faces():
    x = np.linspace(-1, 1, 40)
    X, Y = np.meshgrid(x, x)
    return [
        (X, Y, np.full_like(X, h_2)),      # 顶面
        (X, Y, np.full_like(X, -h_2)),     # 底面
        (X, np.full_like(X, -1), Y*h_2),   # 前侧面
        (X, np.full_like(X, 1), Y*h_2),    # 后侧面
        (np.full_like(X, -1), X, Y*h_2),   # 左侧面
        (np.full_like(X, 1), X, Y*h_2)     # 右侧面
    ]

# -------------------- 物理运动学变形场 --------------------
# 【A 矩阵】：纯面内伸缩 (拉伸X，同时伴随Y和Z向的泊松收缩)
def def_A(X, Y, Z):
    return X*(1+0.18), Y*(1-0.08), Z*(1-0.08)

# 【D 矩阵】：纯面外弯曲 (基于严谨的 Kirchhoff-Love 板理论)
def def_D(X, Y, Z):
    kappa = 0.4
    # u = -z*kappa*x (上表面受压，下表面受拉), w = 0.5*kappa*x^2
    return X - Z*kappa*X, Y, Z + 0.5*kappa*X**2

# 【B 矩阵】：复杂拉伸-扭转耦合 (模拟具备面内斜向不对称的超材料)
def def_B(X, Y, Z):
    eps = 0.25    # 宏观面内拉伸应变
    nu = 0.1      # 宏观泊松收缩
    k = 0.45      # 耦合激发的扭转曲率 (马鞍面)
    # 面内伸缩 + 扭转引起的切向与法向位移的严谨叠加
    Xd = X * (1 + eps) - Z * k * Y
    Yd = Y * (1 - nu) - Z * k * X
    Zd = Z + k * X * Y
    return Xd, Yd, Zd

# -------------------- 渲染与导出模块 --------------------
def generate_matrix_plot(mode, filename):
    # 将6cm转换为英寸以精确控制物理输出尺寸
    cm = 1 / 2.54
    fig = plt.figure(figsize=(6*cm, 6*cm))
    ax = fig.add_subplot(111, projection='3d')

    # 1. 绘制基准参照系：原始实体透视线框
    for ex, ey, ez in get_edges():
        ax.plot(ex, ey, ez, color='gray', linestyle='--', lw=0.6, alpha=0.5)

    deform_funcs = {'A': def_A, 'B': def_B, 'D': def_D}
    func = deform_funcs[mode]
    color = morandi_colors[mode]

    # 2. 绘制变形后的 3D 实体闭合曲面
    for fx, fy, fz in get_faces():
        X_def, Y_def, Z_def = func(fx, fy, fz)
        # 去除面内部网格线，依靠 shade=True 赋予真实 CAD 光影感
        ax.plot_surface(X_def, Y_def, Z_def, color=color, edgecolor='none', alpha=0.95, shade=True)

    # 3. 勾勒变形实体的锐利轮廓线
    for ex, ey, ez in get_edges():
        X_def, Y_def, Z_def = func(ex, ey, ez)
        ax.plot(X_def, Y_def, Z_def, color='#666666', lw=0.5, alpha=0.9)

    # 统一三维坐标轴视野范围，确保3张图的缩放比例绝对一致
    ax.set_xlim([-1.6, 1.6])
    ax.set_ylim([-1.6, 1.6])
    ax.set_zlim([-0.9, 0.9])

    # 彻底关闭坐标系和边框
    ax.axis('off')

    # 消除 matplotlib 默认白边，最大化利用 6cm 空间
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # 导出高分辨率透明底图片
    plt.savefig(filename, dpi=600, bbox_inches='tight', pad_inches=0.01, transparent=True)
    plt.close()

# -------------------- 执行批量生成 --------------------
if __name__ == '__main__':
    print("正在渲染 [A] 矩阵纯拉伸实体模型...")
    generate_matrix_plot('A', 'Matrix_A_Stretch.png')

    print("正在渲染 [B] 矩阵拉扭耦合实体模型...")
    generate_matrix_plot('B', 'Matrix_B_TensionTorsion.png')

    print("正在渲染 [D] 矩阵纯弯曲实体模型...")
    generate_matrix_plot('D', 'Matrix_D_Bending.png')

    print("全部生成完毕！图片已保存在当前代码运行目录下。")

正在渲染 [A] 矩阵纯拉伸实体模型...
正在渲染 [B] 矩阵拉扭耦合实体模型...
正在渲染 [D] 矩阵纯弯曲实体模型...
全部生成完毕！图片已保存在当前代码运行目录下。
